# 01 — Documents

**LLMEvalIQ v1 — Phases 2 & 3**

Phase 2 picks the corpus and proves it is usable.
Phase 3 turns it into the gold evaluation set (`data/evaluation/questions.csv`).

Ground truth is the foundation of everything downstream: Recall@K and
correctness are both meaningless without it.

## Setup

In [ ]:
# Run this first in every notebook.
# Notebooks live in notebooks/, but our code lives in src/ — this makes
# `from src.ingestion import ...` work by putting the project root on the path.
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

DOCUMENTS_DIR = PROJECT_ROOT / "data" / "documents"
EVALUATION_DIR = PROJECT_ROOT / "data" / "evaluation"

print("Project root:", PROJECT_ROOT)
print("Documents   :", DOCUMENTS_DIR)

## Phase 2.1 — Inspect the corpus

Never trust a folder of PDFs you have not measured. Run this **before**
writing a single evaluation question.

In [ ]:
"""Inspect every PDF in data/documents/ before trusting any of it.

Why this cell exists: the single most common way a RAG project dies is a
corpus of scanned PDFs. They look identical to real PDFs in Finder, but they
are images — pypdf extracts ~0 characters and every downstream metric is
garbage. `chars_per_page` catches that in 5 seconds instead of 5 hours.
"""
import pandas as pd
from pypdf import PdfReader

rows = []
for pdf_path in sorted(DOCUMENTS_DIR.glob("*.pdf")):
    reader = PdfReader(pdf_path)
    n_pages = len(reader.pages)
    # `or ""` guards against pages that yield None (images, blank pages).
    text = "\n".join(page.extract_text() or "" for page in reader.pages)
    rows.append({
        "document": pdf_path.name,
        "pages": n_pages,
        "characters": len(text),
        "chars_per_page": round(len(text) / n_pages) if n_pages else 0,
    })

corpus = pd.DataFrame(rows)
corpus

### Interpreting `chars_per_page`

| Value | Meaning |
|---|---|
| 1,500 - 3,500 | Healthy text PDF ✅ |
| 200 - 1,000 | Sparse — heavy tables/figures. Usable, watch it. |
| < 200 | **Scanned image.** pypdf cannot read it. Replace or OCR. ❌ |

A scanned PDF is the single most common way this project fails silently:
retrieval returns nothing useful and every metric reads as a model problem
when it is really a data problem.

## Phase 2.2 — Health check against the targets

In [ ]:
"""Corpus health check.

Note on the page target: the original plan said 200-500 pages. Real corpora of
full annual reports run far larger, and that turned out to be FINE — extra
pages are distractor content, which makes retrieval genuinely hard and
Recall@K meaningful. What actually matters is (a) enough DOCUMENTS that the
random-retrieval baseline is low, (b) clean text, (c) no single document
dominating the index. Pages are reported as information, not pass/fail.

Changing a threshold because reality disagreed is legitimate — as long as you
say why. Silently moving a goalpost is not.
"""
from src.ingestion import CHUNK_SIZE, CHUNK_OVERLAP

if corpus.empty:
    raise SystemExit("No PDFs found. Put them in data/documents/ first.")

n_docs = len(corpus)
total_pages = int(corpus["pages"].sum())
total_chars = int(corpus["characters"].sum())

# A text PDF yields ~1,500-6,500 chars/page. Under ~200 means a scan.
suspect = corpus[corpus["chars_per_page"] < 200]

# No single document should dominate — otherwise headline scores mostly
# describe how the pipeline handles THAT document.
share = corpus["characters"] / total_chars
biggest = corpus.loc[share.idxmax(), "document"]

def check(label: str, ok: bool, detail: str) -> None:
    print(f"{'PASS' if ok else 'FAIL'}  {label:<20} {detail}")

print("CORPUS HEALTH CHECK")
print("-" * 64)
check("Document count", 10 <= n_docs <= 20, f"{n_docs} (target 10-20)")
check("Text extraction", suspect.empty,
      "all readable" if suspect.empty else f"{len(suspect)} suspect: {list(suspect['document'])}")
check("Index balance", share.max() < 0.25,
      f"largest is {share.max():.0%} ({biggest}) — want <25%")
print("-" * 64)

stride = CHUNK_SIZE - CHUNK_OVERLAP
print(f"INFO  Total pages          {total_pages:,}")
print(f"INFO  Total characters     {total_chars:,}")
print(f"INFO  Estimated chunks     ~{total_chars // stride:,}  (stride {stride} chars)")
print(f"INFO  One-time embed cost  ~${total_chars / 4 * 1e-6 * 0.02:.3f}")

## Phase 2.3 — Write the manifest

The manifest is your corpus documentation. Its `document` column is the exact
string `questions.csv` will use in `relevant_document` — the join key that
Recall@K scores against.

In [ ]:
"""Write a manifest: what each document is, so questions.csv can reference it.

The `document` column here becomes the `relevant_document` column in
questions.csv. They must match EXACTLY — that string is the join key the
Recall@K metric depends on.
"""
manifest_path = DOCUMENTS_DIR / "MANIFEST.csv"
manifest = corpus[["document", "pages"]].copy()
manifest["title"] = ""        # e.g. "Siemens Annual Report 2024"
manifest["entity"] = ""       # e.g. "Siemens AG"
manifest["fiscal_year"] = ""  # e.g. 2024
manifest["source_url"] = ""   # where you downloaded it — for reproducibility

manifest.to_csv(manifest_path, index=False)
print(f"Wrote {manifest_path}")
print("Now open it and fill in the blank columns by hand.")
manifest

---

# Phase 3 — Gold Evaluation Set

Everything downstream is scored against `data/evaluation/questions.csv`.
It is the ruler. If the ruler is wrong, every measurement is wrong — and
nothing in the pipeline will tell you.

**Schema (v2)**

| Column | Meaning |
|---|---|
| `question_id` | Stable unique key, e.g. `q001`. Never reuse or renumber. |
| `question` | What a real user would ask, in their words. |
| `reference_answer` | The correct answer, written by you from the document. |
| `relevant_document` | Exact PDF filename. Join key for **Document Recall@K**. |
| `relevant_pages` | `42`, `42-44`, or `42;47`. For human debugging. |
| `evidence_text` | Verbatim snippet containing the answer. Drives **Evidence Recall@K**. |
| `difficulty` | `easy` / `medium` / `difficult` |
| `question_type` | `numeric` / `entity` / `date` / `list` / `comparison` / `descriptive` |

**Why `evidence_text` exists:** our largest report is 590 pages ≈ 4,100 chunks.
Document-level recall counts *any* of those as a hit, including the auditor's
report. Evidence recall asks the honest question — was the passage carrying
the answer actually retrieved?

**Target mix:** 60 easy, 30 medium, 10 difficult.

## Phase 3.0 — Find candidate passages

Locates fact-carrying sentences with page numbers so you can write questions
without scrolling hundreds of pages. **You** write the question and verify the
answer — that judgement is what makes the gold set worth anything.

In [ ]:
"""Candidate finder — a DRAFTING AID, not a question generator.

It surfaces passages that carry hard facts, with page numbers, so you do not
have to scroll 590 pages hunting for something askable. You still write the
question and verify the answer: that judgement is the whole value of the gold
set, and the one part a model must not do for you.

Why not let an LLM write the questions? It would phrase them using the
document's own wording, so retrieval would succeed on keyword overlap instead
of meaning (trap 1). Every metric would read high and mean nothing.
"""
import re
from pypdf import PdfReader

# Front matter (AGM notices, e-voting instructions) is fact-dense and useless.
BOILER = re.compile(r"\b(AGM|e-?voting|SEBI|MCA|Circular|Regulation|weblink|"
    r"Registrar|depositor|RTA|scrutinizer|postal ballot|IEPF|proxy|CIN|"
    r"email|website)\b", re.I)
# Business substance we actually want questions about.
SIGNAL = re.compile(r"\b(revenue|turnover|profit|loss|sales|income|margin|EBITDA|"
    r"dividend|employee|capacity|export|production|segment|subsidiary|"
    r"net worth|earnings per share|EPS)\b", re.I)
ABBREV = re.compile(r"\\b(Rs|No|Mr|Mrs|Ms|Dr|Ltd|Pvt|Co|Inc|St|vs|etc|i\\.e|e\\.g)\\.", re.I)
NUM = re.compile(r"(₹|Rs\.?|\bcrore\b|\blakh\b|\blacs?\b|\bmillion\b|%)", re.I)


def find_candidates(document: str, limit: int = 25) -> list[tuple[int, str]]:
    """Return (page_number, sentence) pairs worth building a question from."""
    reader = PdfReader(DOCUMENTS_DIR / document)
    n_pages = len(reader.pages)
    out: list[tuple[int, str]] = []
    for i in range(int(n_pages * 0.12), n_pages):   # skip front matter
        text = re.sub(r"\s+", " ", reader.pages[i].extract_text() or "")
        for sentence in re.split(r"(?<=[.;])\s+", text):
            s = sentence.strip().replace("\u2024", ".")
            # Ceiling is 400, not 240: protecting abbreviations (above) merges
            # fragments into longer sentences, and a tight ceiling then drops
            # exactly the fully-labelled passages we most want.
            if not 70 <= len(s) <= 400:      continue
            if BOILER.search(s):             continue
            if not SIGNAL.search(s):         continue
            if not NUM.search(s):            continue
            if sum(c.isdigit() for c in s) < 4: continue
            out.append((i + 1, s))
            if len(out) >= limit:
                return out
    return out


DOCUMENT = "ecoplast_2025_26_annual_report.pdf"   # <- change per document
for page, sentence in find_candidates(DOCUMENT):
    print(f"p.{page:<5}{sentence[:150]}")
    print()

## Phase 3.1 — Validate the gold set

Run this after every edit. It is cheap; a wrong ruler is not.

In [ ]:
"""Validate the gold evaluation set (two-file schema).

questions.csv  — one row per question
evidence.csv   — one row per REQUIRED PASSAGE; a question may have several

Run this after every edit. A broken gold set does not raise an error during
evaluation — it silently produces wrong scores.
"""
import pandas as pd
from src.evaluation import QUESTION_TYPES, DIFFICULTIES

questions = pd.read_csv(EVALUATION_DIR / "questions.csv", dtype=str).fillna("")
evidence  = pd.read_csv(EVALUATION_DIR / "evidence.csv",  dtype=str).fillna("")
problems: list[str] = []

Q_COLS = ["question_id", "question", "reference_answer", "difficulty", "question_type"]
E_COLS = ["evidence_id", "question_id", "relevant_document", "relevant_pages", "evidence_text"]

for df, cols, name in ((questions, Q_COLS, "questions"), (evidence, E_COLS, "evidence")):
    for c in cols:
        if c not in df.columns:
            problems.append(f"{name}.csv missing column '{c}'")
        elif (df[c].str.strip() == "").any():
            rows = df.index[df[c].str.strip() == ""].tolist()
            problems.append(f"{name}.csv blank '{c}' at rows {rows[:5]}")

if questions["question_id"].duplicated().any(): problems.append("duplicate question_id")
if evidence["evidence_id"].duplicated().any():  problems.append("duplicate evidence_id")

bad_d = sorted(set(questions["difficulty"]) - set(DIFFICULTIES))
bad_t = sorted(set(questions["question_type"]) - QUESTION_TYPES)
if bad_d: problems.append(f"unknown difficulty {bad_d}")
if bad_t: problems.append(f"unknown question_type {bad_t}")

# Referential integrity, BOTH directions. A question with no evidence row is
# unscoreable for retrieval and would silently vanish from Phase 5.
orphans = sorted(set(evidence["question_id"]) - set(questions["question_id"]))
no_ev   = sorted(set(questions["question_id"]) - set(evidence["question_id"]))
if orphans: problems.append(f"evidence points at unknown question: {orphans[:5]}")
if no_ev:   problems.append(f"questions with NO evidence: {no_ev[:5]}")

on_disk = {p.name for p in DOCUMENTS_DIR.glob("*.pdf")}
missing = sorted(set(evidence["relevant_document"]) - on_disk)
if missing: problems.append(f"evidence document not on disk: {missing[:5]}")

short = evidence.index[evidence["evidence_text"].str.len() < 40].tolist()
if short: problems.append(f"evidence_text under 40 chars at rows {short[:5]} — quote a fuller sentence")

per_q = evidence.groupby("question_id").size()

print(f"GOLD SET — {len(questions)} questions, {len(evidence)} evidence passages")
print("-" * 66)
for level in DIFFICULTIES:
    print(f"  {level:<11}{int((questions['difficulty'] == level).sum()):>3}")
print("\n  evidence passages per question:")
for n, c in per_q.value_counts().sort_index().items():
    print(f"    {n} passage(s): {c:>3} question(s)")
print("-" * 66)
print("No problems found." if not problems else
      f"{len(problems)} PROBLEM(S):\n  " + "\n  ".join(problems))

## Phase 3.2 — Coverage

Check that questions are spread across the corpus rather than clustered in
one or two documents.

In [ ]:
"""How are questions spread across documents and difficulty?

Concentration is a silent bias. If 40 of your 100 questions come from one
report, your headline scores mostly describe how well the pipeline handles
THAT report — not your corpus.
"""
if not questions.empty:
    coverage = (
        questions.pivot_table(
            index="relevant_document",
            columns="difficulty",
            values="question_id",
            aggfunc="count",
            fill_value=0,
        )
        .reindex(columns=["easy", "medium", "difficult"], fill_value=0)
    )
    coverage["total"] = coverage.sum(axis=1)
    coverage = coverage.sort_values("total", ascending=False)

    display(coverage)

    share = coverage["total"] / coverage["total"].sum()
    print(f"\nMost-represented document: {share.max():.0%} of all questions")
    if share.max() > 0.25:
        print("WARNING: over 25% from one document — spread the questions wider.")
else:
    print("questions.csv is empty — nothing to plot yet.")

## Phase 3.3 — Verify evidence snippets

The check that decides whether Evidence Recall@K can work at all.

In [ ]:
"""Verify every evidence_text appears in the extracted PDF text.

THE most important check in Phase 3. Evidence metrics work by substring
matching; a snippet that was mistyped, or copied from the PDF viewer rather
than the extracted text, matches NOTHING — and every affected question scores
as a retrieval failure that is really a transcription error in your ruler.

Real example from this corpus: Ecoplast renders the apostrophe as U+0027,
Goa Carbon as U+2019. Same word, different byte, silent failure.
"""
from pypdf import PdfReader
from src.evaluation import normalise

cache: dict[str, str] = {}
def extracted(doc: str) -> str:
    if doc not in cache:
        reader = PdfReader(DOCUMENTS_DIR / doc)
        cache[doc] = normalise("\n".join(p.extract_text() or "" for p in reader.pages))
    return cache[doc]

failures = []
for _, row in evidence.iterrows():
    doc = row["relevant_document"]
    if not (DOCUMENTS_DIR / doc).exists():
        failures.append((row["evidence_id"], row["question_id"], "document missing"))
    elif normalise(row["evidence_text"]) not in extracted(doc):
        failures.append((row["evidence_id"], row["question_id"], row["evidence_text"][:60]))

print(f"EVIDENCE VERIFICATION — {len(evidence)} checked, {len(failures)} failed")
print("-" * 66)
for eid, qid, detail in failures:
    print(f"  x {eid} ({qid}): {detail}...")
if failures:
    print("\nFix by re-copying from the cell 3.0 output, NOT the PDF viewer.")
    print("Do not loosen the matcher — that would make coverage pass on")
    print("evidence that was never actually retrieved.")
else:
    print("All evidence snippets located. Ruler is sound.")